# M3 — Keypoints de Ruedas y Conteo de Ejes (YOLO11-pose)

In [1]:
!nvidia-smi -L || echo "Activa GPU T4 en Entorno de ejecucion > Cambiar tipo de entorno"
!pip -q install -U ultralytics==8.4.218 roboflow==1.7.4 pyyaml==6.0.4 pandas matplotlib requests
# CLIP solo se instala si se ejecuta preetiquetado con YOLOE desde cero
# !pip -q install 'git+https://github.com/ultralytics/CLIP.git'

GPU 0: NVIDIA A40 (UUID: GPU-e5aa9d4d-cd18-12ae-f2a6-9724225d43f6)


In [2]:
import os, glob, json, shutil, random, yaml, cv2, requests, numpy as np, pandas as pd, torch
from pathlib import Path
from getpass import getpass
from roboflow import Roboflow
from ultralytics import YOLO, YOLOE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
WORKSPACE = 'cristopher-diego-rodriguez-ayala'
PROJECT_M3 = 'peaje-llantas-keypoints'
USAR_DATASET_ROBOFLOW_PREPARADO = True  # Descarga directo la version ya curada y aumentada en Roboflow
KPT_NOMBRES = ['centro', 'contacto', 'superior', 'izquierda', 'derecha']
ESQUELETO_1BASED = [[1, 2], [1, 3], [1, 4], [1, 5]]
FLIP_IDX = [0, 1, 2, 4, 3]

UMBRAL_CONTENCION = 0.7
UMBRAL_CERCANA = 0.65
SEPARACION_EJE = 0.6

ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY') or getpass('Pega tu Roboflow Private API Key: ')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
print('Dispositivo:', DEVICE)

Dispositivo: cuda


In [3]:
BASE_M3 = Path('/content/m3_fuentes')
COCO_M3 = Path('/content/m3_coco_upload')
YOLO_M3 = Path('/content/m3_yolo_pose')
for d in (BASE_M3, COCO_M3, YOLO_M3):
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True, exist_ok=True)

def descargar_rf(ws_slug, proj_slug, ver, fmt, dest):
    try:
        ds = rf.workspace(ws_slug).project(proj_slug).version(ver).download(fmt, location=str(dest))
        print('Descargado:', proj_slug, '->', ds.location)
        return ds
    except Exception as e:
        print('Aviso descarga', proj_slug, e)
        return None

if USAR_DATASET_ROBOFLOW_PREPARADO:
    ds_w1 = ds_w2 = ds_bo = None
    print('Modo rapido activo: se descargara directamente la version curada de keypoints desde Roboflow.')
else:
    ds_w1 = descargar_rf('datacluster-labs-agryi', 'vehicle-wheel-detection', 1, 'yolov8', BASE_M3 / 'wheel_multi')
    ds_w2 = descargar_rf('faisal-sduw7', 'vehicle-wheel', 9, 'yolov8', BASE_M3 / 'wheel_general')
    ds_bo = descargar_rf('lab2e3', 'transporte-publico-bolivia', 3, 'yolov8', BASE_M3 / 'bolivia')

    WIKI_M3 = BASE_M3 / 'wikimedia'
    WIKI_M3.mkdir(parents=True, exist_ok=True)
    headers = {'User-Agent': 'IAM811-PeajeEducativo/1.0 (https://unifranz.edu.bo)'}
    for q in ('Semi truck side view', 'Cargo truck 3 axle side', 'Bus side view'):
        try:
            r = requests.get('https://commons.wikimedia.org/w/api.php', params={
                'action': 'query', 'generator': 'search', 'gsrsearch': q, 'gsrnamespace': 6,
                'gsrlimit': 12, 'prop': 'imageinfo', 'iiprop': 'url|mime', 'iiurlwidth': 960, 'format': 'json'
            }, headers=headers, timeout=15).json()
            for pid, pinfo in ((r.get('query') or {}).get('pages') or {}).items():
                ii = (pinfo.get('imageinfo') or [{}])[0]
                url = ii.get('thumburl') or ii.get('url')
                if url and ('jpeg' in ii.get('mime', '') or 'png' in ii.get('mime', '')):
                    resp = requests.get(url, headers=headers, timeout=15)
                    if resp.status_code == 200 and len(resp.content) > 8000:
                        (WIKI_M3 / f'wiki_{pid}.jpg').write_bytes(resp.content)
        except Exception as e:
            print('Aviso Wikimedia:', q, e)

Modo rapido activo: se descargara directamente la version curada de keypoints desde Roboflow.


In [4]:
items_m3 = []
if USAR_DATASET_ROBOFLOW_PREPARADO:
    print('Modo rapido: se descargara el dataset M3 con 5 keypoints desde Roboflow.')
else:
    yoloe = YOLOE('yoloe-11s-seg.pt')
    yoloe.set_classes(['wheel', 'tire'], yoloe.get_text_pe(['wheel', 'tire']))

    def kps_de_mascara(mask):
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not cnts:
            return None
        c = max(cnts, key=cv2.contourArea)
        if cv2.contourArea(c) < 10:
            return None
        pts = c.reshape(-1, 2).astype(float)
        if len(pts) >= 5:
            (cx, cy), _, _ = cv2.fitEllipse(c)
        else:
            cx, cy = pts.mean(0)
        return np.array([
            [cx, cy],
            pts[pts[:, 1].argmax()],
            pts[pts[:, 1].argmin()],
            pts[pts[:, 0].argmin()],
            pts[pts[:, 0].argmax()],
        ], dtype=float)

    def kps_de_caja(x1, y1, x2, y2):
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        return np.array([[cx, cy], [cx, y2], [cx, y1], [x1, cy], [x2, cy]], dtype=float)

    def recolectar_de_dataset_cajas(ds_obj, target_names, max_imgs=None):
        if ds_obj is None:
            return []
        ds_path = ds_obj.location
        y = yaml.safe_load((Path(ds_path) / 'data.yaml').read_text())
        raw_names = y['names']
        names = {i: n for i, n in enumerate(raw_names)} if isinstance(raw_names, list) else {int(k): v for k, v in raw_names.items()}
        items = []
        for sp in ('train', 'valid', 'test'):
            idir = Path(ds_path) / sp / 'images'
            if not idir.exists():
                continue
            sp_out = 'valid' if sp in ('valid', 'test') else 'train'
            for ip in sorted(idir.glob('*')):
                lp = ip.parent.parent / 'labels' / f'{ip.stem}.txt'
                if not lp.exists():
                    continue
                im = cv2.imread(str(ip))
                if im is None:
                    continue
                H, W = im.shape[:2]
                llantas = []
                for line in lp.read_text().splitlines():
                    p = line.strip().split()
                    if len(p) < 5:
                        continue
                    cname = str(names[int(float(p[0]))]).strip().lower()
                    if cname not in target_names:
                        continue
                    cx, cy, bw, bh = [float(v) for v in p[1:5]]
                    x1, y1 = max(0.0, (cx - bw/2) * W), max(0.0, (cy - bh/2) * H)
                    x2, y2 = min(float(W), (cx + bw/2) * W), min(float(H), (cy + bh/2) * H)
                    if x2 - x1 < 8 or y2 - y1 < 8:
                        continue
                    llantas.append(((x1, y1, x2, y2), kps_de_caja(x1, y1, x2, y2)))
                if llantas:
                    items.append((ip, sp_out, llantas))
        if max_imgs and len(items) > max_imgs:
            random.seed(42); random.shuffle(items)
            items = items[:max_imgs]
        return items

    items_m3 += recolectar_de_dataset_cajas(ds_w1, {'wheel'})
    items_m3 += recolectar_de_dataset_cajas(ds_w2, {'0', 'wheel', 'vehicle wheel detection - v1 2024-06-01 4-18am'}, max_imgs=360)

    extra_imgs = (sorted(Path(ds_bo.location).rglob('images/*')) if ds_bo else []) + sorted(WIKI_M3.glob('*.jpg'))
    for i, ip in enumerate(extra_imgs[:50]):
        im = cv2.imread(str(ip))
        if im is None:
            continue
        H, W = im.shape[:2]
        r = yoloe.predict(im, conf=0.18, imgsz=640, device=DEVICE, agnostic_nms=True, verbose=False)[0]
        llantas = []
        if r.masks is not None and len(r.boxes):
            for b, poly in zip(r.boxes.xyxy.cpu().numpy(), r.masks.xy):
                if len(poly) < 3:
                    continue
                m = np.zeros((H, W), np.uint8)
                cv2.fillPoly(m, [poly.astype(np.int32)], 1)
                k = kps_de_mascara(m)
                if k is not None:
                    x1, y1 = min(float(b[0]), float(k[:, 0].min())), min(float(b[1]), float(k[:, 1].min()))
                    x2, y2 = max(float(b[2]), float(k[:, 0].max())), max(float(b[3]), float(k[:, 1].max()))
                    llantas.append(((x1, y1, x2, y2), k))
        if llantas:
            items_m3.append((ip, 'valid' if i % 5 == 0 else 'train', llantas))

    print('Total de imagenes con llantas y 5 keypoints:', len(items_m3))

Modo rapido: se descargara el dataset M3 con 5 keypoints desde Roboflow.


In [5]:
ws = rf.workspace(WORKSPACE)
proj = ws.project(PROJECT_M3)
versiones = proj.versions()

if USAR_DATASET_ROBOFLOW_PREPARADO and versiones:
    ver_id = max(int(v.version) for v in versiones)
    print('Descargando version Roboflow Keypoints existente:', ver_id)
    ds_rf_m3 = proj.version(ver_id).download('yolov8', location='/content/m3_roboflow_final')
    DATA_YAML_M3 = str(Path(ds_rf_m3.location) / 'data.yaml')
    cfg3 = yaml.safe_load(Path(DATA_YAML_M3).read_text())
    cfg3['kpt_shape'] = [5, 3]
    cfg3['flip_idx'] = FLIP_IDX
    Path(DATA_YAML_M3).write_text(yaml.safe_dump(cfg3, sort_keys=False))
else:
    for sp in ('train', 'valid'):
        (YOLO_M3 / 'images' / sp).mkdir(parents=True, exist_ok=True)
        (YOLO_M3 / 'labels' / sp).mkdir(parents=True, exist_ok=True)
        (COCO_M3 / sp).mkdir(parents=True, exist_ok=True)

    coco_m3 = {
        sp: {
            'info': {'year': '2026', 'version': '1', 'description': 'IAM-811 M3 keypoints'},
            'licenses': [{'id': 1, 'url': '', 'name': 'MIT'}],
            'images': [],
            'annotations': [],
            'categories': [{
                'id': 1, 'name': 'llanta', 'supercategory': 'rueda',
                'keypoints': KPT_NOMBRES, 'skeleton': ESQUELETO_1BASED,
            }],
        }
        for sp in ('train', 'valid')
    }
    ann_id = 1
    total_llantas = 0

    for idx, (src_p, sp, llantas) in enumerate(items_m3):
        im = cv2.imread(str(src_p))
        if im is None:
            continue
        H, W = im.shape[:2]
        fname = f'{idx:05d}_{src_p.stem}.jpg'
        cv2.imwrite(str(YOLO_M3 / 'images' / sp / fname), im)
        cv2.imwrite(str(COCO_M3 / sp / fname), im)
        coco_m3[sp]['images'].append({'id': idx + 1, 'file_name': fname, 'width': W, 'height': H})

        yolo_lines = []
        for (x1, y1, x2, y2), k in llantas:
            bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
            vals = [(x1 + x2) / 2 / W, (y1 + y2) / 2 / H, bw / W, bh / H]
            kps_coco = []
            for kx, ky in k:
                kx_c, ky_c = min(max(float(kx), 0.0), float(W)), min(max(float(ky), 0.0), float(H))
                vals += [kx_c / W, ky_c / H, 2]
                kps_coco += [round(kx_c, 2), round(ky_c, 2), 2]
            yolo_lines.append('0 ' + ' '.join(f'{v:.6f}' if isinstance(v, float) else str(v) for v in vals))
            coco_m3[sp]['annotations'].append({
                'id': ann_id, 'image_id': idx + 1, 'category_id': 1,
                'bbox': [round(x1, 2), round(y1, 2), round(bw, 2), round(bh, 2)],
                'area': round(bw * bh, 2), 'iscrowd': 0,
                'keypoints': kps_coco, 'num_keypoints': 5,
            })
            ann_id += 1
            total_llantas += 1
        (YOLO_M3 / 'labels' / sp / f'{idx:05d}_{src_p.stem}.txt').write_text('\n'.join(yolo_lines))

    for sp in ('train', 'valid'):
        (COCO_M3 / sp / '_annotations.coco.json').write_text(json.dumps(coco_m3[sp]))

    DATA_YAML_M3 = str(YOLO_M3 / 'data.yaml')
    Path(DATA_YAML_M3).write_text(
        f'path: {YOLO_M3}\ntrain: images/train\nval: images/valid\nkpt_shape: [5, 3]\nflip_idx: {FLIP_IDX}\nnames:\n  0: llanta\n'
    )
    print('Llantas totales anotadas:', total_llantas)
    try:
        ws.upload_dataset(dataset_path=str(COCO_M3), project_name=PROJECT_M3, project_type='keypoint-detection', num_workers=8)
        ver_id = proj.generate_version(settings={
            'preprocessing': {'auto-orient': True, 'resize': {'width': 640, 'height': 640, 'format': 'Stretch to'}},
            'augmentation': {
                'image': {'versions': 2},
                'flip': {'horizontal': True, 'vertical': False},
                'brightness': {'brighten': True, 'darken': True, 'percent': 15},
            },
        })
        ds_rf_m3 = proj.version(int(ver_id)).download('yolov8', location='/content/m3_roboflow_final')
        DATA_YAML_M3 = str(Path(ds_rf_m3.location) / 'data.yaml')
        cfg3 = yaml.safe_load(Path(DATA_YAML_M3).read_text())
        cfg3['kpt_shape'] = [5, 3]
        cfg3['flip_idx'] = FLIP_IDX
        Path(DATA_YAML_M3).write_text(yaml.safe_dump(cfg3, sort_keys=False))
    except Exception as exc:
        print('Aviso subida/versionado Roboflow (entrenando con copia local):', exc)

loading Roboflow workspace...


loading Roboflow project...


Descargando version Roboflow Keypoints existente: 1


Extracting Dataset Version Zip to /content/m3_roboflow_final in yolov8::   0%|          | 0/1852 [00:00<?, ?it/s]

Extracting Dataset Version Zip to /content/m3_roboflow_final in yolov8::  25%|██▌       | 472/1852 [00:00<00:00, 4719.66it/s]

Extracting Dataset Version Zip to /content/m3_roboflow_final in yolov8::  53%|█████▎    | 983/1852 [00:00<00:00, 4946.76it/s]

Extracting Dataset Version Zip to /content/m3_roboflow_final in yolov8::  96%|█████████▌| 1769/1852 [00:00<00:00, 6275.21it/s]

Extracting Dataset Version Zip to /content/m3_roboflow_final in yolov8:: 100%|██████████| 1852/1852 [00:00<00:00, 5951.40it/s]

In [6]:
modelo_pose = YOLO('yolo11s-pose.pt')
modelo_pose.train(
    data=DATA_YAML_M3,
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    device=0 if DEVICE == 'cuda' else 'cpu',
    project='/content/runs/pose',
    name='m3_llantas',
    exist_ok=True,
)

Ultralytics 8.4.218 🚀 Python-3.12.3 torch-2.8.0+cu129 CUDA:0 (NVIDIA A40, 48544MiB)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/m3_roboflow_final/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=m3_llantas, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, perspective=0.0, plots=Tr

Overriding model.yaml kpt_shape=[17, 3] with kpt_shape=[5, 3]


Overriding model.yaml nc=80 with nc=1



                   from  n    params  module                                       arguments                     


  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 


... (2,740 outputs left out)

                   all        146        341      0.895       0.71      0.781      0.576      0.886      0.751      0.797      0.711


Speed: 0.1ms preprocess, 15.4ms inference, 0.0ms loss, 1.5ms postprocess per image


Results saved to /workspace/colab_work/runs/pose/m3_llantas


ultralytics.utils.metrics.PoseMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78efea533dd0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(P)', 'F1-Confidence(P)', 'Precision-Confidence(P)', 'Recall-Confidence(P)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    

In [7]:
BEST_M3 = Path('/content/runs/pose/m3_llantas/weights/best.pt')
pose_model = YOLO(str(BEST_M3))
val_pose = pose_model.val(data=DATA_YAML_M3)

def altura_visible(k):
    return max(1.0, float(k[1, 1] - k[2, 1]))

def contar_ejes_kps(kps_list):
    if not kps_list:
        return 0
    hmax = max(altura_visible(k) for k in kps_list)
    cercanas = [k for k in kps_list if altura_visible(k) >= UMBRAL_CERCANA * hmax]
    xs = sorted(float(k[0, 0]) for k in cercanas)
    diam = float(np.median([altura_visible(k) for k in cercanas]))
    return max(1, 1 + sum(1 for p, q in zip(xs, xs[1:]) if q - p > SEPARACION_EJE * diam))

ds_root_m3 = Path(DATA_YAML_M3).parent
val_imgs_dir = ds_root_m3 / 'valid' / 'images'
if not val_imgs_dir.exists():
    val_imgs_dir = ds_root_m3 / 'images' / 'valid'
val_lbls_dir = val_imgs_dir.parent.parent / 'labels' / val_imgs_dir.name if val_imgs_dir.parent.name == 'images' else val_imgs_dir.parent / 'labels'

total_imgs_m3 = 0
total_llantas_m3 = 0
for lp in ds_root_m3.rglob('labels/*.txt'):
    lines = [ln for ln in lp.read_text().splitlines() if ln.strip()]
    if lines:
        total_imgs_m3 += 1
        total_llantas_m3 += len(lines)

errores_ejes = []
for src_p in sorted(val_imgs_dir.glob('*')):
    lbl_p = val_lbls_dir / f'{src_p.stem}.txt'
    im_v = cv2.imread(str(src_p))
    if im_v is None or not lbl_p.exists():
        continue
    Hv, Wv = im_v.shape[:2]
    gt_kps = []
    for ln in lbl_p.read_text().splitlines():
        ps = [float(v) for v in ln.strip().split()]
        if len(ps) >= 20:
            k = np.array([[ps[5 + 3*i] * Wv, ps[6 + 3*i] * Hv, 1.0] for i in range(5)], dtype=float)
            gt_kps.append(k)
    if not gt_kps:
        continue
    gt_ejes = contar_ejes_kps(gt_kps)
    r = pose_model.predict(im_v, conf=0.25, device=DEVICE, verbose=False)[0]
    pred_kps = []
    if r.keypoints is not None and len(r.boxes):
        xy = r.keypoints.xy.cpu().numpy()
        kc = r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None else np.ones(xy.shape[:2])
        for p, c in zip(xy, kc):
            pred_kps.append(np.concatenate([p, c[:, None]], 1))
    pred_ejes = contar_ejes_kps(pred_kps)
    errores_ejes.append(abs(pred_ejes - gt_ejes))

metricas_m3 = {
    'modelo': 'yolo11s-pose',
    'total_imagenes': total_imgs_m3,
    'total_llantas': total_llantas_m3,
    'kpt_shape': [5, 3],
    'box_map50': round(float(val_pose.box.map50), 4),
    'pose_map50': round(float(val_pose.pose.map50), 4),
    'pose_map50_95': round(float(val_pose.pose.map), 4),
    'ejes_mae_valid': round(float(np.mean(errores_ejes)), 4) if errores_ejes else None,
    'ejes_exactitud_valid': round(float(np.mean([e == 0 for e in errores_ejes])), 4) if errores_ejes else None,
}
print('=== METRICAS M3 ===')
print(json.dumps(metricas_m3, indent=2, ensure_ascii=False))
Path('/content/resumen_metricas_m3.json').write_text(json.dumps(metricas_m3, indent=2, ensure_ascii=False))
shutil.copy(BEST_M3, '/content/best_m3_pose.pt')

import sys, platform
print('=== DIAGNOSTICO COLAB M3 ===')
print(json.dumps({
    'python': sys.version.split()[0], 'os': platform.platform(),
    'torch': torch.__version__, 'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, indent=2))

try:
    from google.colab import files
    files.download('/content/best_m3_pose.pt')
    files.download('/content/resumen_metricas_m3.json')
except ImportError:
    print('Ejecucion fuera de Colab: artefactos guardados en /content')

Ultralytics 8.4.218 🚀 Python-3.12.3 torch-2.8.0+cu129 CUDA:0 (NVIDIA A40, 48544MiB)


YOLO11s-pose summary (fused): 109 layers, 9,700,560 parameters, 0 gradients, 22.4 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 943.3±207.4 MB/s, size: 62.4 KB)


val: Scanning /workspace/colab_work/m3_roboflow_final/valid/labels.cache... 146 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 146/146 55.7Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 10% ━─────────── 1/10 2.4s/it 0.7s<21.3s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 20% ━━────────── 2/10 1.9it/s 0.9s<4.3s

... (5 outputs left out)

                   all        146        341      0.895       0.71      0.781      0.575      0.883      0.748      0.797      0.711


Speed: 0.8ms preprocess, 3.5ms inference, 0.0ms loss, 1.0ms postprocess per image


Results saved to /tmp/runs/pose/val


=== METRICAS M3 ===
{
  "modelo": "yolo11s-pose",
  "total_imagenes": 924,
  "total_llantas": 2023,
  "kpt_shape": [
    5,
    3
  ],
  "box_map50": 0.7806,
  "pose_map50": 0.7968,
  "pose_map50_95": 0.7113,
  "ejes_mae_valid": 0.226,
  "ejes_exactitud_valid": 0.7945
}
=== DIAGNOSTICO COLAB M3 ===
{
  "python": "3.12.3",
  "os": "Linux-6.8.0-149-generic-x86_64-with-glibc2.39",
  "torch": "2.8.0+cu129",
  "cuda": true,
  "gpu": "NVIDIA A40"
}
Ejecucion fuera de Colab: artefactos guardados en /content
